# CenterLoRA-ProtBERT v1 — backbone screen and frozen test

This notebook replaces only the ESM-2 backbone of the v1 development champion with ProtBERT-BFD. It runs seed 42 for all 15 epochs on the identical protein-grouped development split. The screen itself does not access the independent test. After a passing decision, the final guarded section can evaluate the frozen development-best checkpoint once on the independent test. That checkpoint was fitted on 82,535 records, with 9,188 development records withheld. A GPU runtime is required.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Clone or update the private repository

Private-repository access uses a fine-grained read-only GitHub token stored in Colab Secrets as `GITHUB_TOKEN`. An optional `HF_TOKEN` avoids unauthenticated Hugging Face rate limits. Notebook access is enabled for each configured secret.

In [ ]:
from pathlib import Path
import os
import subprocess

try:
    from google.colab import userdata
    try:
        GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
    except Exception:
        GITHUB_TOKEN = None
    try:
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        HF_TOKEN = None
except ImportError:
    GITHUB_TOKEN = None
    HF_TOKEN = None

git_environment = os.environ.copy()
git_environment['GIT_TERMINAL_PROMPT'] = '0'
askpass_path = Path('/tmp/ubipred_git_askpass.sh')
if GITHUB_TOKEN:
    askpass_path.write_text(
        '#!/bin/sh\n'
        'case "$1" in\n'
        '  *Username*) printf "%s\\n" "x-access-token" ;;\n'
        '  *) printf "%s\\n" "$GITHUB_TOKEN" ;;\n'
        'esac\n',
        encoding='utf-8',
    )
    askpass_path.chmod(0o700)
    git_environment['GIT_ASKPASS'] = str(askpass_path)
    git_environment['GITHUB_TOKEN'] = GITHUB_TOKEN

REPO_ROOT = Path('/content/Ubiquitination-Sites-Prediction')
try:
    if not (REPO_ROOT / '.git').exists():
        subprocess.run([
            'git', 'clone', '--recurse-submodules',
            'https://github.com/kaleabnega/Ubiquitination-Sites-Prediction.git',
            str(REPO_ROOT),
        ], check=True, env=git_environment)
    else:
        subprocess.run(
            ['git', '-C', str(REPO_ROOT), 'pull', '--ff-only'],
            check=True, env=git_environment,
        )
        subprocess.run(
            ['git', '-C', str(REPO_ROOT), 'submodule', 'update', '--init', '--recursive'],
            check=True, env=git_environment,
        )
except subprocess.CalledProcessError as error:
    raise RuntimeError(
        'Git access failed. Add GITHUB_TOKEN to Colab Secrets for a private repository.'
    ) from error
finally:
    askpass_path.unlink(missing_ok=True)

os.chdir(REPO_ROOT)
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['HF_HOME'] = str(
    Path('/content/drive/MyDrive/MMUbiPred-research/huggingface-cache')
)
Path(os.environ['HF_HOME']).mkdir(parents=True, exist_ok=True)
print('Project:', REPO_ROOT)
print('Persistent Hugging Face cache:', os.environ['HF_HOME'])
subprocess.run(['git', 'rev-parse', 'HEAD'], check=True)

In [ ]:
%pip install -q -r experiment/requirements-colab.txt
%pip uninstall -y torchao

In [ ]:
import json
import platform
import sys
import peft
import sklearn
import torch
import transformers

print('Python:', sys.version)
print('Platform:', platform.platform())
print('scikit-learn:', sklearn.__version__)
print('PyTorch:', torch.__version__)
print('Transformers:', transformers.__version__)
print('PEFT:', peft.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    properties = torch.cuda.get_device_properties(0)
    print('GPU:', properties.name)
    print('GPU memory (GiB):', round(properties.total_memory / 2**30, 2))
else:
    raise RuntimeError('A GPU runtime is required for this screen.')

def run_live(command):
    command = [str(part) for part in command]
    print('$', ' '.join(command), flush=True)
    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    if process.stdout is None:
        raise RuntimeError('Could not open subprocess output stream')
    for line in process.stdout:
        print(line, end='', flush=True)
    return_code = process.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, command)

## Preflight tests

In [ ]:
run_live([
    sys.executable, '-u', '-m', 'unittest', 'discover',
    '-s', 'experiment/tests',
])

## Run or resume the seed-42 screen

ProtBERT-BFD is approximately 1.68 GB to download and is substantially slower than ESM-2 35M. The model cache and run artifacts are stored in Drive. A lightweight trainable-state checkpoint is saved after every completed epoch, allowing continuation from the next epoch after a runtime interruption.

In [ ]:
RUN_DIR = Path(
    '/content/drive/MyDrive/MMUbiPred-research/experiments/'
    'center-lora-protbert-v1-seed42'
)
RUN_DIR.mkdir(parents=True, exist_ok=True)
METRICS_PATH = RUN_DIR / 'validation_metrics.json'
RESUME_PATH = RUN_DIR / 'development_resume.pt'

if METRICS_PATH.exists():
    print('Completed metrics already exist; training was skipped:', METRICS_PATH)
else:
    if RESUME_PATH.exists():
        resume_state = torch.load(RESUME_PATH, map_location='cpu', weights_only=False)
        print('Resuming after completed epoch:', resume_state['completed_epoch'])
        del resume_state
    run_live([
        sys.executable, '-u', 'experiment/scripts/train.py',
        '--config', 'experiment/configs/center_lora_protbert_screen_v1.json',
        '--seed', '42',
        '--output-dir', RUN_DIR,
        '--development-only',
        '--resume-development',
    ])
print('Persistent run directory:', RUN_DIR)

## Compare against the frozen ESM-2 v1 champion

The comparison requires the completed full-15 seed-42 v1 result and rejects mismatched validation split hashes.

In [ ]:
V1_METRICS_PATH = Path(
    '/content/drive/MyDrive/MMUbiPred-research/benchmarks/'
    'center-lora-esm2-full15-three-seed-v1/'
    'center_lora_esm2_full15_v1/seed_42/validation_metrics.json'
)

def valid_full15_v1_metrics(path):
    try:
        metrics = json.loads(path.read_text())
        manifest = json.loads((path.parent / 'run_manifest.json').read_text())
        config = manifest['config']
        return (
            int(config['seed']) == 42
            and int(config['epochs']) == 15
            and config['experiment_name'] == 'center_lora_esm2_full15_v1'
            and metrics['development_selection']['epochs_completed'] == 15
            and not manifest['locked_test_accessed']
        )
    except Exception:
        return False

if not valid_full15_v1_metrics(V1_METRICS_PATH):
    archive_root = Path('/content/drive/MyDrive/MMUbiPred-research')
    matches = [
        path for path in archive_root.rglob('validation_metrics.json')
        if path != METRICS_PATH and valid_full15_v1_metrics(path)
    ]
    if not matches:
        raise FileNotFoundError(
            'No complete full-15 seed-42 ESM-2 v1 result was found on Drive.'
        )
    V1_METRICS_PATH = max(matches, key=lambda path: path.stat().st_mtime)

v1_metrics = json.loads(V1_METRICS_PATH.read_text())
protbert_metrics = json.loads(METRICS_PATH.read_text())
v1_manifest = json.loads((V1_METRICS_PATH.parent / 'run_manifest.json').read_text())
protbert_manifest = json.loads((METRICS_PATH.parent / 'run_manifest.json').read_text())

v1_hash = v1_manifest['development_split']['validation_indices_sha256']
protbert_hash = protbert_manifest['development_split']['validation_indices_sha256']
if v1_hash != protbert_hash:
    raise RuntimeError(
        f'Validation split mismatch: ESM-2={v1_hash}, ProtBERT={protbert_hash}'
    )

display({'v1_metrics_path': str(V1_METRICS_PATH), 'validation_hash': v1_hash})
display({
    'protbert_fixed_threshold': protbert_metrics['fixed_threshold'],
    'protbert_selected_threshold': protbert_metrics['validation_selected_threshold'],
    'protbert_development_selection': protbert_metrics['development_selection'],
    'protbert_branch_names': protbert_metrics['branch_names'],
    'protbert_prediction_diagnostics': protbert_metrics['prediction_diagnostics'],
})

In [ ]:
metric_keys = ['mcc', 'accuracy', 'sensitivity', 'specificity', 'auroc', 'auprc']
comparison = {
    metric: {
        'esm2_v1': float(v1_metrics['fixed_threshold'][metric]),
        'protbert_v1': float(protbert_metrics['fixed_threshold'][metric]),
        'protbert_minus_esm2': (
            float(protbert_metrics['fixed_threshold'][metric])
            - float(v1_metrics['fixed_threshold'][metric])
        ),
    }
    for metric in metric_keys
}
comparison['selected_mcc'] = {
    'esm2_v1': float(v1_metrics['validation_selected_threshold']['mcc']),
    'protbert_v1': float(protbert_metrics['validation_selected_threshold']['mcc']),
    'protbert_minus_esm2': (
        float(protbert_metrics['validation_selected_threshold']['mcc'])
        - float(v1_metrics['validation_selected_threshold']['mcc'])
    ),
}
display(comparison)

mcc_gain = comparison['mcc']['protbert_minus_esm2']
selected_mcc_gain = comparison['selected_mcc']['protbert_minus_esm2']
auroc_gain = comparison['auroc']['protbert_minus_esm2']
auprc_gain = comparison['auprc']['protbert_minus_esm2']
collapsed = bool(
    protbert_metrics['prediction_diagnostics']['single_class_predictions_at_0_5']
)

passed = (
    not collapsed
    and mcc_gain >= 0.005
    and selected_mcc_gain >= 0
    and auroc_gain >= -0.002
    and auprc_gain >= -0.002
)
if passed:
    decision = 'ADVANCE: ProtBERT passed every predeclared condition and is eligible for three-seed confirmation.'
else:
    decision = 'REJECT: retain CenterLoRA-ESM2 v1 and close this architecture screen.'

print(decision)
print('Locked independent test accessed during screen:', False)

## One-time independent-test evaluation of the existing seed-42 model

This stage uses `development_best.pt`, the epoch-4 checkpoint that produced the reported development metrics. It does not retrain or refit the model. The primary threshold is 0.5, and the secondary threshold 0.555 was selected on development data. Execution closes architecture exploration. The evaluator refuses to overwrite an existing test result.

In [ ]:
ACKNOWLEDGE_FINAL_LOCKED_TEST = False
LOCKED_METRICS_PATH = RUN_DIR / 'locked_test_metrics.json'
DEVELOPMENT_CHECKPOINT_PATH = RUN_DIR / 'development_best.pt'

if LOCKED_METRICS_PATH.exists():
    print('Existing one-time independent-test result was not rerun.')
    locked_metrics = json.loads(LOCKED_METRICS_PATH.read_text())
    display(locked_metrics)
elif ACKNOWLEDGE_FINAL_LOCKED_TEST:
    if not passed:
        raise RuntimeError('The predeclared development screen did not pass.')
    if not DEVELOPMENT_CHECKPOINT_PATH.exists():
        raise FileNotFoundError(DEVELOPMENT_CHECKPOINT_PATH)
    run_live([
        sys.executable, '-u', 'experiment/scripts/evaluate.py',
        '--run-dir', RUN_DIR,
        '--checkpoint-name', 'development_best.pt',
        '--data-dir', 'replication/MMUbiPred',
        '--batch-size', '32',
        '--num-workers', '0',
        '--allow-locked-test',
    ])
    locked_metrics = json.loads(LOCKED_METRICS_PATH.read_text())
    display(locked_metrics)
else:
    print('Locked test was not accessed. Set ACKNOWLEDGE_FINAL_LOCKED_TEST=True when ready.')

## Direct comparison with the reproduced paper model

In [ ]:
if LOCKED_METRICS_PATH.exists():
    locked_metrics = json.loads(LOCKED_METRICS_PATH.read_text())
    display({
        'protbert_seed42_fixed_0.5': locked_metrics['fixed_threshold'],
        'protbert_seed42_dev_selected_0.555': locked_metrics['validation_selected_threshold'],
        'reproduced_paper_model': locked_metrics['paper_baseline'],
        'fixed_0.5_difference_from_paper': locked_metrics['fixed_threshold_difference_from_paper'],
        'training_scope': locked_metrics['training_scope'],
    })
else:
    print('Run the acknowledged one-time evaluation before displaying final results.')